# Практическое ДЗ 1. Удаление фона с помощью SVD

В этом практическом ДЗ мы познакомимся с одним из возможных приложений сингулярного разложения &mdash; удаление фона из видео.
Для этого сначала загрузим видео, на котором есть движущиеся объекты и неизменный фон.


**Замечание: далее пользоваться циклами запрещено, если это явно не прописано в задании. Вместо этого используйте функции numpy.**

In [ ]:
%pip install -q moviepy==1.0.3
%pip install -q ffmpeg

In [ ]:
pip install imageio-ffmpeg

In [ ]:
import moviepy.editor as mpe
video = mpe.VideoFileClip("/Users/ekaterinaseloceva/Downloads/prac1_final/SVD_video_low.mp4")
video.ipython_display(width=300, maxduration=250)

Импортируем необходимые библиотеки и представим **цветное** видео в виде четырехмерного массива размеров
`(#кадров) x (height) x (width) x 3`, где последние 3 координаты соответствуют каналам RGB.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
video.fps

In [ ]:
import numpy as np

def video_to_array(video, dtype=np.float32):
    """Convert MoviePy VideoFileClip to a numpy array of shape (nframes, H, W, 3).

    Returns float array in [0, 1] when dtype is floating, otherwise uint array in [0, 255].
    """
    duration = float(video.duration)
    fps = float(video.fps)
    nframes = int(round(fps * duration))
    size_w, size_h = video.size  # moviepy gives (W, H)

    arr = np.zeros((nframes, size_h, size_w, 3), dtype=dtype)

    for i in range(nframes):
        t = min(i / fps, duration - 1e-6)   # avoid hitting exactly the end
        frame = video.get_frame(t)[..., :3] # RGB (drop alpha if present)

        if np.issubdtype(dtype, np.floating):
            arr[i] = frame.astype(dtype) / 255.0
        else:
            arr[i] = frame.astype(dtype)

    print(f'image size: {size_w} x {size_h}, \nnumber of frames: {nframes}')
    return arr


In [ ]:
arr = video_to_array(video)
plt.imshow(arr[0])
plt.title('The first frame of the video')
plt.axis('off')

### a. Применение базового алгоритма SVD (30 баллов)

1. **(10 балла)** С помощью функций `np.reshape(...), np.transpose(...)` получите из четырехмерного массива `arr`
двумерный массив `M` размера `(size_h * size_w * 3) x nframes`, каждым столбцом которого является цветное изображение
размера `size_h x size_w x 3`, представленное в виде вектора длины `size_h * size_w * 3`.


In [ ]:
nframes, size_h, size_w, n_channels = arr.shape
size_h1, size_w1 = size_h, size_w
M = np. transpose(arr). reshape( (-1, nframes), order='f')

plt.figure(figsize=(10, 6))
plt.imshow(M, cmap='gray', aspect='auto')
plt.xlabel('Frame index')
plt.ylabel('Pixel index (flattened RGB)')
plt.title('Matrix M (each column is a frame)')

Если всё сделано правильно, то вы сможете восстановить первый кадр из первого столбца:

In [ ]:
first_frame = M[:, 0].reshape(size_h, size_w, n_channels)# YOUR CODE GOES HERE
plt.imshow(np.clip(first_frame, 0, 1))
plt.title('The first frame of the video')
plt.axis('off')

2. **(6 баллов)** Вычислите сингулярное разложение матрицы ```M```. Постройте график сингулярных чисел, нормированных на $\|A\|_2$. Используйте логарифмическую шкалу. Для этого, например, используйте функцию ```plt.semilogy```. Объясните, почему наибольшее и несколько наименьших (близких к 0) сингулярных число заметно отличается от остальных.

**Замечание:** При построении графиков величин с отличающимися на порядки значениями полезно использовать логарифмическую шкалу. Чтобы убедиться в этом, попробуйте построить график не в логарифмической шкале; из него будет фактически невозможно понять характер убывания сингулярных чисел. Не забывайте подписывать оси!

In [ ]:
U, S, V_T = np.linalg.svd(M, full_matrices=False)
S_norm = S / S[0]

plt.figure(figsize=(10, 6))
plt.semilogy(S_norm, 'pink', linewidth=2)
plt.xlabel(r'Индекс сингулярного числа $i$')
plt.ylabel(r'$\sigma_i / \|A\|_2$')
plt.title(r'Нормированные сингулярные числа $\sigma_i / \|A\|_2$ (Логарифмическая шкала)')
plt.grid(True, alpha=0.3)
plt.show()

Большое различие между наибольшими и наименьшими сингулярными числами обусловлено тем, что на видео есть много повторяющейся информации (например, одинаковый фон на каждом кадре), а эту информацию как раз и описывают первые сингулярные числа. Они «общие» для многих кадров, поэтому их вклад огромен.

Последние сингулярные числа, наоборот, описывают шум и случайности, которые уникальны для каждого отдельного кадра — дрожание камеры, зернистость. Эти компоненты не повторяются, поэтому их вклад в общую структуру видео ничтожно мал

3. **(10 баллов)** Напишите функцию trunc_svd(M, r), которая считает оптимальное приближение $M_{r}$ двумерного массива $M$ заданного ранга, а также относительную точность этого приближения в фробениусовой норме, т.е.
$$
\frac{\|M - M_{r}\|_F}{\|M\|_F}.
$$
Для расчета относительной точности используйте только сингулярные числа матрицы $M$.

In [ ]:
def trunc_svd(M, r):
    '''
        Input
            M: 2D numpy array
            r: rank value for truncation

        Output
            Mr: 2D numpy array of the same size as M
            rel_eps: relative error of rank-r approximation Mr in Frobenius norm
    '''
    U, S, V_T = np.linalg.svd(M, full_matrices=False)
    M_r = U[:, :r] @ np.diag(S[:r]) @ V_T[:r, :]
    norm_M = np.sqrt(np.sum(S**2))
    norm_M_M_r = np.sqrt(np.sum(S[r:]**2))
    return M_r, norm_M_M_r / norm_M


4. **(4 балла)** Используя написанную функцию, найдите наилучшее приближение матрицы `M` матрицей `M_svd` ранга 1.
Постройте изображения первого кадра:
- исходного видео,
- фона (из приближения ранга 1),
- движущихся объектов (разность исходного кадра и фона).


In [ ]:
M_svd, approximation = trunc_svd(M, 1)
print(f'Наилучшее приближение матрицы M матрицей M_svd ранга 1: {approximation}')

In [ ]:
_, axs = plt.subplots(1, 3, figsize=(15, 6))

axs[0].imshow(np.clip(M[:, 0].reshape(size_h, size_w, n_channels), 0, 1))# YOUR CODE GOES HERE
axs[0].set_title("Исходное изображение")
axs[0].axis('off')

axs[1].imshow(np.clip(M_svd[:, 0].reshape(size_h, size_w, n_channels), 0, 1))# YOUR CODE GOES HERE
axs[1].set_title("Фон (ранг 1)")
axs[1].axis('off')

axs[2].imshow(np.clip(M[:, 0].reshape(size_h, size_w, n_channels) - 
                      M_svd[:, 0].reshape(size_h, size_w, n_channels), 0, 1))# YOUR CODE GOES HERE
axs[2].set_title("Движущиеся объекты")
axs[2].axis('off')

### b. Применение рандомизированного алгоритма SVD (28 баллов)

Загрузим теперь видео в более высоком разрешении.

In [ ]:
import moviepy.editor as mpe
video3 = mpe.VideoFileClip("/Users/ekaterinaseloceva/Downloads/prac1_final/SVD_video.mp4", target_resolution=(100, 178))
video3.ipython_display(width=300, maxduration=250)
# arr3 will be created below

In [ ]:
arr3 = video_to_array(video3)
nframes3, size_h3, size_w3, n_channels3 = arr3.shape
M3 = np.transpose(arr3).reshape(-1, nframes3, order='F')

Использование функции ```np.linalg.svd``` является эффективным для относительно небольших массивов из-за быстрого роста сложности алгоритма в зависимости от размера матрицы. Используем рандомизированный алгоритм из лекций для ускорения вычислений (есть также [пост](https://gregorygundersen.com/blog/2019/01/17/randomized-svd/) с описанием алгоритма).

1. **(16 баллов)** Реализуйте рандомизированный алгоритм SVD из лекции, который аппроксимирует матрицу с заданным рангом $r$ (алгоритм запускается с ```r + oversampling``` случайных векторов, после чего ранг обрезается до ```r``` с наименьшей ошибкой). Убедитесь, что вы не вычисляете в явном виде матрицу $QQ^\top$. Если на заданной матрице алгоритм работает слишком долго (минуты), то возможно, вы что-то делаете не так.

In [ ]:
def rand_svd(M, r, oversampling=10):
    '''
        Input
            M: 2D numpy array
            r: rank value for truncation
            oversampling: number of extra random vectors to approximate range(M)

        Output
            Mr: 2D numpy array of rank r and of the same size as M
            rel_eps: relative error of rank-r approximation Mr
    '''
    Omega = np.random.randn(M.shape[1], r + oversampling)
    Y = M @ Omega
    Q, R = np.linalg.qr(Y)
    U, S, Vh = np.linalg.svd(Q.T @ M, full_matrices=False)
    Mr = (Q @ U)[:, :r] @ np.diag(S[:r]) @ Vh[:r, :]
    rel_eps = np.linalg.matrix_norm(M - Mr) / np.linalg.matrix_norm(M)
    return Mr, rel_eps

2. **(2 балла)** Используя `rand_svd`, найдите наилучшее приближение матрицы `M3` матрицей `M3_rand` ранга 1.
Постройте изображения первого кадра:
- исходного видео,
- фона (из приближения ранга 1),
- движущихся объектов (разность исходного кадра и фона).


In [ ]:
M3_rand, approximation = trunc_svd(M3, 1)
print(f'Наилучшее приближение матрицы M3 матрицей M3_rand ранга 1: {approximation}')

In [ ]:
_, axs = plt.subplots(1, 3, figsize=(15, 6))

axs[0].imshow(np.clip(M3[:, 0].reshape(size_h3, size_w3, n_channels3), 0, 1))
axs[0].set_title("Исходное изображение")
axs[0].axis('off')

axs[1].imshow(np.clip(M3_rand[:, 0].reshape(size_h3, size_w3, n_channels3), 0, 1))
axs[1].set_title("Фон (rand SVD, ранг 1)")
axs[1].axis('off')

moving = M3[:, 0].reshape(size_h3, size_w3, n_channels3) - \
         M3_rand[:, 0].reshape(size_h3, size_w3, n_channels3)
axs[2].imshow(np.clip(moving, 0, 1))
axs[2].set_title("Движущиеся объекты")
axs[2].axis('off')

3. **(10 баллов)** Постройте график функции
$$
\frac{||M_{rand}(p) - M||_F}{||M||_F}
$$
при $r=2$ в зависимости от $p$ (```oversampling=p``` в функции ```rand_svd```). По $p$ выберите сетку $[0, 30]$ с шагом 2.
Так как $M_{rand}(p)$ получено с помощью рандомизированного алгоритма, усредните Ваш результат, запустив алгоритм 10 раз.
При построении графика используйте логарифмическую шкалу по оси с ошибкой.
**В данном задании разрешается использовать циклы для перебора по сетке и запуска алгоритма 10 раз.**

In [ ]:
import time
## YOUR CODE HERE
r = 2
p = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
avg_errors = []
for i in p:
    result_rel_eps = 0
    for j in range(10):
        M_rand_p, rel_eps = rand_svd(M3, r, oversampling=i)
        result_rel_eps += rel_eps
    result_rel_eps /= 10
    avg_errors.append(result_rel_eps)

plt.figure(figsize=(10, 6))
plt.semilogy(p, avg_errors, color='pink', marker='o', linewidth=2, markersize=8)
plt.xlabel('oversampling = p')
plt.ylabel(r'Относительная ошибка $\|M_{rand}(p) - M\|_F / \|M\|_F$')
plt.title(f'Зависимость ошибки от p при r=2 (усреднено по 10 запускам)')
plt.grid(True, alpha=0.2)
plt.xticks(p)
plt.show()

**Замечание:** ```np.linalg.svd``` на этих размерах матриц и значенях рангов будет работать заметно медленнее (правда, с полностью контролируемой точностью), чем рандомизированный алгоритм. Также обратите внимание, что если не указать опцию ```full_matrices=False``` в ```np.linalg.svd```, то на данном примере может случиться переполнение по памяти.

### c. Монохромное видео (10 баллов)

В этой секции вы изучите, как палитра изображения связана со структурой соответствующего тензора

In [ ]:
import moviepy.editor as mpe
video_mono = mpe.VideoFileClip("/Users/ekaterinaseloceva/Downloads/prac1_final/SVD_video_mono.mp4")
video_mono.ipython_display(width=300, maxduration=250)

In [ ]:
arr = video_to_array(video_mono)
plt.imshow(arr[0])
plt.title('The first frame of the video')
plt.axis('off')

Как вы могли заметить, цвета в текущем изображении различаются сильно меньше, чем в изображениях предыдущих пунктов.

Проверьте, как баланс цветов связан с корнем из стабильного ранга $\rho = \dfrac{\| A\|_F^2} {\| A\|_2^2}$, где матрица $A$ - развертка исходного тензора по моде цветов (каналов). В частности, посмотрите на отношение $$\dfrac{\sqrt{\rho_{\text{orig}}} - 1}{\sqrt{\rho_{\text{mono}}} - 1}$$

In [ ]:
arr_mono = video_to_array(video_mono)
nframes, size_h, size_w, n_channels = arr_mono.shape

In [ ]:
def unfolding_stable_rank(video_array):
    f'''
        Input
            video_array: 4D numpy array nframes x size_h x size_w x n_channels

        Output
            stable rank of unfolding by 4-th mode
    '''
    nframes, h, w, c = video_array.shape
    A = np.transpose(video_array, (3, 0, 1, 2)).reshape(c, -1)
    U, S, V_T = np.linalg.svd(A, full_matrices=False)
    A_F = np.sqrt(np.sum(S**2))
    A_2 = S[0]
    p = (A_F**2) / (A_2**2)
    return p

In [ ]:
orig_image_rank = unfolding_stable_rank(arr3)
warm_image_rank = unfolding_stable_rank(arr_mono)

In [ ]:
##YOUR CODE HERE
result = (np.sqrt(orig_image_rank) - 1) / (np.sqrt(warm_image_rank) - 1)
print(f'Стабильный ранг для оригинального изображения: {orig_image_rank}')
print(f'Стабильный ранг для мнохромного изображения: {warm_image_rank}')
print(f'Результат соотношения: {result}')

Если вы все сделали правильно, то значение для оригинального изображения должно получиться больше, чем для монохромного. Кроме того, для монохромного эффективный ранг должен быть немного больше единицы. Почему так получилось?

Эффективный ранг для монохромного изображения получился чуть больше единицы (1.0002259016036987), потому что в теории у монохромного видео все три цветовых канала должны быть одинаковыми, и ранг должен быть ровно 1. Но на практике из-за шума при съемке, округлений чисел и сжатия видео появляются микроскопические различия между каналами, поэтому ранг получается немного больше единицы. Для оригинального цветного изображения эффективный ранг тоже близок к единице (1.003700613975525), но всё же заметно больше, потому что здесь каналы реально разные — они несут разную цветовую информацию, а не просто шум. Отношение получилось около 16.37, что как раз показывает, насколько цветовая информация сильнее, чем просто шум

### d. Видео с переменным освещением (12 баллов)

Загрузим теперь более интересное видео, в котором со временем меняется освещение (можно, к примеру, представить, что встаёт солнце).

In [ ]:
video2 = mpe.VideoFileClip("/Users/ekaterinaseloceva/Downloads/prac1_final/SVD_video_bright.mp4")
video2.ipython_display(width=300, maxduration=250)

In [ ]:
arr2 = video_to_array(video2)

Пока что возьмём лишь первые 80% видео, остальной частью воспользуемся позже.

In [ ]:
breakpoint = round(0.8*arr2.shape[0])
remaining = arr2[breakpoint:]
arr2 = arr2[:breakpoint]

1. Аналогично пункту a.1) получите из массива `arr2` матрицу `M2` размера `(size_h * size_w * 3) x nframes2`.


In [ ]:
nframes2, size_h, size_w, n_channels2 = arr2.shape
M2 = np.transpose(arr2).reshape(-1, nframes2, order='F')

plt.figure(figsize=(10, 6))
plt.imshow(M2, cmap='gray', aspect='auto')
plt.xlabel('Frame index')
plt.ylabel('Pixel index (flattened RGB)')
plt.title('Matrix M2')

2. **(3 балла)** Примените методы из пунктов a) и b) (то есть библиотечное полное SVD и рандомизированное SVD) для получения наилучшего приближения ранга 2 (чтобы также учесть изменение освещения) для матрицы `M2`. Сравните время работы алгоритмов.

In [ ]:
import time

start = time.time()
U, s, Vt = np.linalg.svd(M2, full_matrices=False)
M2_svd = U[:, :2] @ np.diag(s[:2]) @ Vt[:2, :]
svd_time = time.time() - start
print(f"Полное SVD: {svd_time:.4f} сек")

start = time.time()
M2_rand, rel_eps = rand_svd(M2, r=2, oversampling=10)
rand_time = time.time() - start
print(f"Рандомизированное SVD: {rand_time:.4f} сек")
print(f"Рандомизированное быстрее в {svd_time/rand_time:.1f} раз")

3. **(4 балла)** Cравните относительные точности таких приближений для настоящего SVD и рандомизированного алгоритма; также сравните их с соответствующей величиной для видео с постоянным освещением. Какие выводы можно сделать?

In [ ]:
M_svd, svd_eps = trunc_svd (M3, 2)
M_rand, rand_eps = rand_svd (M3, 2)
M2_svd, svd2_eps = trunc_svd (M2, 2)
M2_rand, rand2_eps = rand_svd (M2, 2)
print(f'M3 svd: {svd_eps}, M3 rand_svd: {rand_eps}')
print(f'M2 svd: {svd2_eps}, M2 rand_svd: {rand2_eps}')

Сравнила точность обычного SVD и рандомизированного SVD для двух видео: цветного с нормальным светом и монохромного, где освещение меняется. Для цветного видео обычный SVD дал ошибку 0.126, а рандомизированный — 0.130 (разница всего 0.04). Для монохромного видео обычный SVD выдал 0.122, рандомизированный — 0.129 (разница 0.007). Получается, что рандомизированный алгоритм работает почти так же точно, как обычный, отличаясь всего на полпроцента-процент. Зато он должен работать быстрее (мы это в предыдущем пункте видели). Еще заметила, что ошибка для обоих видео примерно одинаковая (около 12-13%), то есть ранг 2 одинаково хорошо (или одинаково плохо) описывает и цветное видео, и монохромное с меняющимся освещением

4. **(5 баллов)** Заполните пропуски в функции `M_to_video`, преобразующей матрицу типа ```M2``` обратно в видео.

In [ ]:
def M_to_video(M, fps, size_w, size_h):
    """Convert matrix M (vectorized frames as columns) back to a MoviePy clip.

    Supports both grayscale (size_h*size_w x nframes) and RGB (size_h*size_w*3 x nframes).
    Assumes floats are in [0,1]; will clip and convert to uint8 for MoviePy.
    """
    nframes = M.shape[1]

    arr_rgb = M.T.reshape(nframes, size_h, size_w, 3)
    def make_frame(t):
        index = min(int(t * fps), nframes - 1)
        frame = arr_rgb[index]

        if frame.dtype.kind == 'f':
            frame = np.clip(frame, 0.0, 1.0)
            frame = (frame * 255).astype(np.uint8)
        else:
            frame = np.clip(frame, 0, 255).astype(np.uint8)

        return frame

    return mpe.VideoClip(make_frame, duration=nframes / fps)


Посмотрим, как выглядит предлагаемое приближение.

In [ ]:
video2_svd = M_to_video(M2_svd - M2, 20, size_w, size_h)
video2_svd.ipython_display(width=300, maxduration=250, fps=20)

Для большей наглядности можете также запустить видео из пункта b) с более высоким разрешением.

### e. Обработка новых кадров (15 баллов)

Предположим, что на камеру поступил новый поток кадров. Мы могли бы увеличить нашу матрицу M2 и пересчитать SVD, но это слишком вычислительно сложно ради нескольких новых кадров. Более того, логично предположить, что если у нас уже было достаточно много кадров в матрице M2, то сингулярные векторы не изменятся сильно от добавления новых кадров.

При этом просто вычесть фон не получится, ведь мы хотим также учитывать изменение освещения. Для этого посчитаем ортогональную проекцию нового кадра на образ матрицы M2.




1. **(5 баллов)** Используя SVD разложение ортогонально спроецируйте новый кадр на образ матрицы M2.

Ваш код должен работать как для 1 кадра (вектора длины size_h * size_w), так и для нескольких (матрицы размера (size_h * size_w) x k). Сложность итогового алгоритма для обработки 1 кадра должна быть O(size_h * size_w). SVD матрицы M2 считайте предпосчитанным.

In [ ]:
new_frame = remaining[-1].reshape(-1)
plt.imshow(remaining[-1])
plt.title("A new frame (from remaining)")
plt.axis('off')

In [ ]:
# YOUR CODE GOES HERE (duplicate code from trunc_SVD)
U, S, V_T = np.linalg.svd(M2, full_matrices=False)
U_2 = U[:, :2]
S_2 = S[:2]
VT_2 = V_T[:2, :]

In [ ]:
def project_new_frames(U, S, VT, new_frames):
    '''
        Input
            U, S, VT: rank r compact SVD of matrix M2 (U @ S @ VT = M2_r)
            new_frames: vector (m,) or matrix (m x k), where m = size_h*size_w*3

        Output
            proj: projection of new frames to Im(M2_r)
    '''
    p = U @ (U.T @ new_frames)
    return p


In [ ]:
new_frame_proj = project_new_frames(U_2, S_2, VT_2, new_frame)

_, axs = plt.subplots(1, 3, figsize=(15, 6))
axs[0].imshow(new_frame.reshape(size_h, size_w, 3)) # YOUR CODE GOES HERE
axs[0].set_title("Исходное изображение")
axs[0].axis('off')

axs[1].imshow(new_frame_proj.reshape(size_h, size_w, 3))  # YOUR CODE GOES HERE
axs[1].set_title("Фон (проекция)")
axs[1].axis('off')

moving_objects = new_frame - new_frame_proj
axs[2].imshow(moving_objects.reshape(size_h, size_w, 3)) # YOUR CODE GOES HERE
axs[2].set_title("Движущиеся объекты")
axs[2].axis('off')

2. **(5 баллов)** Используя `np.concatenate` добавьте кадры из `remaining` к столбцам `M2`, чтобы получить матрицу полного видео `M_full`.
Аналогично преобразуйте массив `remaining` в матрицу формата `(size_h * size_w * 3) x remaining_nframes`. Спроецируйте все новые кадры и преобразуйте результат обратно в видео.


In [ ]:
remaining_nframes, size_h_rem, size_w_rem, n_channels_rem = remaining.shape
M_remaining = np.transpose(remaining).reshape(-1, remaining_nframes, order='F')  # YOUR CODE GOES HERE
M_full = np.concatenate((M2, M_remaining), axis=1)  # YOUR CODE GOES HERE


In [ ]:
M_proj = project_new_frames(U_2, S_2, VT_2, M_full)

video2_svd = M_to_video(M_full - M_proj, 20, size_w, size_h)
video2_svd.ipython_display(width=300, maxduration=250, fps=20)

3. **(2 балла)** Как говорилось выше, можно сэкономить много ресурсов с помощью рандомизированного SVD алгоритма. Рассмотрите аналогичную ортопроекцию с помощью рандомизированного SVD разложения.

In [ ]:
r = 2
oversampling = 10

Omega = np.random.randn(M2.shape[1], r + oversampling)
Y = M2 @ Omega
Q, R = np.linalg.qr(Y)
U, S, Vh = np.linalg.svd(Q.T @ M2, full_matrices=False)

U_2_rand = Q @ U[:, :2]
S_2_rand = S[:2]
VT_2_rand = Vh[:2, :]

In [ ]:
M_proj_rand = project_new_frames(U_2_rand, S_2_rand, VT_2_rand, M_full)

video2_svd = M_to_video(M_full - M_proj_rand, 20, size_w, size_h)
video2_svd.ipython_display(width=300, maxduration=250, fps=20)

4. **(3 балла)** Найдите $M_{\text{true}}$ - лучшее приближение 2 ранга матрицы M2_full с помощью честного SVD (аналогично заданию c). Посчитайте относительные точности приближений из пунктов 2 и 3; сравните качество видео. Какие выводы можно сделать по качеству видео и итоговым ошибкам?

$$
    \dfrac{\| M_{\text{true}} - M_{\text{proj}} \|_F}{\| M_{\text{true}} \|_{F}};\quad
    \dfrac{\| M_{\text{true}} - M_{\text{proj rand}} \|_F}{\| M_{\text{true}} \|_{F}}
$$

In [ ]:
M_true = trunc_svd(M_full, 2)[0]

video2_svd = M_to_video(M_full - M_true, 20, size_w, size_h)
video2_svd.ipython_display(width=300, maxduration=250, fps=20)

In [ ]:
# YOUR CODE GOES HERE
error_proj_svd = np.linalg.norm(M_true - M_proj, 'fro') / np.linalg.norm(M_true, 'fro')
error_proj_rand = np.linalg.norm(M_true - M_proj_rand, 'fro') / np.linalg.norm(M_true, 'fro')

print(f"Относительная точность приближения M_proj: {error_proj_svd:.6f}")
print(f"Относительная точность приближения M_proj_rand: {error_proj_rand:.6f}")

**Выводы:**

По результатам видно, что у обоих подходов достаточно маленькая ошибка, для SVD - 3,2%, а для рандомизированного SVD - 4,8%. Разница между ними незначительная, поэтому видео будут выглядеть почти одинаково. При этом как видно в предыдущих выводах, рандомизированный SVD работает значительно быстрее, поэтому для обработки больших данных он подходит больше (совсем немного уступает в точности, но работает кратно быстрее)


### f. Robust PCA (5 баллов)

Заметим, что матрицу $M$ можно приближенно представить в виде $M = L + S$, где $L$ - малоранговая матрица, а $S$ - разреженная (то есть содержащая большое количество нулей). Для поиска $L$ и $S$ мы могли бы попытаться решить задачу

$$
    \mathrm{rank}(L) + \alpha\ \mathrm{nnz}(S) \to \min_{L,S}, \quad \alpha > 0
$$

при ограничении $M = L + S$, где $\mathrm{nnz}(S)$ обозначет число ненулевых элементов (<b>n</b>umber of <b>n</b>on<b>z</b>eros).
Однако такую задачу решать крайне сложно из-за отсутствия непрерывности и выпуклости.
Поэтому обычно ее [заменяют на более простую](https://arxiv.org/pdf/0912.3599.pdf):

$$
    \|L\|_* + \alpha \|S\|_{\mathrm{sum}} \to \min_{L, S}, \quad \alpha > 0,
$$

где $\|\cdot\|_*$ обозначает ядерную норму матрицы, а $\|\cdot\|_{\mathrm{sum}}$ является суммой модулей всех элементов матрицы ($\ell_1$-норма).

Для вычисления robust PCA скачаем код по ссылке [https://github.com/dganguli/robust-pca](https://github.com/dganguli/robust-pca) и импортируем его:

In [ ]:
# Если вы работаете на windows - скачивайте самостоятельно
!wget https://raw.githubusercontent.com/dganguli/robust-pca/master/r_pca.py
import r_pca

Мы использовали класс `R_pca` и запустили функцию `fit(max_iter=4000, iter_print=100)` для нахождения матриц $L$ и $S$ по матрице $M$ из пункта a).

1. **(5 баллов)** Постройте изображения первого кадра:
- исходного видео,
- фона ($L$),
- движущихся объектов ($S$).

Чтобы ускорить расчёты, используйте только четверть временного ряда и берите каждый второй кадр.


In [ ]:
Mt = M[:, :M.shape[1]//4:2]
solver = r_pca.RobustPCA(Mt, mu=0.68)  # в нашем случае такой mu работает лучше исходного
L, S = solver.fit(max_iter=4000, iter_print=100)


_, axs = plt.subplots(1, 3, figsize=(15, 6))
axs[0].imshow(Mt[:, 0].reshape(size_h, size_w, n_channels)) # YOUR CODE GOES HERE
axs[0].set_title("Исходное изображение")
axs[0].axis('off')

axs[1].imshow(L[:, 0].reshape(size_h, size_w, n_channels)) # YOUR CODE GOES HERE
axs[1].set_title("Фон (L)")
axs[1].axis('off')

axs[2].imshow(S[:, 0].reshape(size_h, size_w, n_channels)) # YOUR CODE GOES HERE
axs[2].set_title("Движущиеся объекты (S)")
axs[2].axis('off')

plt.tight_layout()
plt.show()

Аналогично можно посмотреть на видео движущихся объектов:

In [ ]:
video_rpca = M_to_video(np.abs(S), 20, size_w1, size_h1)
video_rpca.ipython_display(width=300, maxduration=250, fps=20)


### g. Бонус

В бонусной части мы познакомимся с более продвинутыми рандомизированными алгоритмами поиска сингулярного разложения. Условие заданий базируется на статье

Halko, Nathan, Per-Gunnar Martinsson, and Joel A. Tropp. "Finding structure with randomness: Probabilistic algorithms for constructing approximate matrix decompositions." SIAM review 53.2 (2011): 217-288.

**Ссылка на статью**: http://users.cms.caltech.edu/~jtropp/papers/HMT11-Finding-Structure-SIREV.pdf

1. **(50 б. баллов)** Покажите, что аналитически (в точной арифметике) алгоритмы  4.3 и 4.4 из статьи, указанной выше, эквивалентны. Все теоретические выкладки приведите в текущем файле с использованием Markdown или прикрепите качественное изображение рукописного текста.

2. **(50 б. баллов)** Реализуйте Алгоритм 4.4 и используйте его для построения приближенного сингулярного разложения матрицы A на матрице из этой ДЗ. Зафиксируйте ранги $r=5, 20, 50$. Для каждого из этих значений на одном рисунке постройте график зависимости нормы (на выбор) разности полученного приближения и оптимального приближения (в выбранной норме) того же ранга от числа q.